In [2]:
using MarineHydro
using PyCall
cpt = pyimport("capytaine")

PyObject <module 'capytaine' from 'C:\\Users\\15183\\.julia\\conda\\3\\x86_64\\Lib\\site-packages\\capytaine\\__init__.py'>

## Create Multi DOF body using Capytaine

In [44]:
# Description of problem
h = Inf # sea depth [m]
omega = 1.03 # frequency [rad/s]
beta = 0 # incident wave angle [rad]
t_DOFs = ["Heave"] # degrees of freedom
r_DOFs = ["Pitch"]
DOFs = [t_DOFs; r_DOFs]

2-element Vector{String}:
 "Heave"
 "Pitch"

In [45]:
# Create Mesh object
radius = 1.5 # 
center = (0.0,0.0,0.0)
len = 2.5
faces_max_radius = 0.5
cptmesh = cpt.meshes.predefined.mesh_horizontal_cylinder(
            radius=radius,
            center=center, 
            length=len, 
            faces_max_radius = faces_max_radius
            ).keep_immersed_part(inplace=true)

PyObject Mesh(vertices=[[... 83 vertices ...]], faces=[[... 80 faces ...]], name="cylinder_15")

In [46]:
omegas = [1.01, 1.05, 1.7]

3-element Vector{Float64}:
 1.01
 1.05
 1.7

In [57]:
# Create FloatingBody object
cptbody = cpt.FloatingBody(mesh=cptmesh)
foreach(dof -> cptbody.add_translation_dof(name=dof), t_DOFs)
foreach(dof -> cptbody.add_rotation_dof(name=dof), r_DOFs)
cptbody.active_dofs = DOFs
cptbody.name = "Body 1"
cptbody.center_of_mass = (0, 0, 0)
cptbody.rotation_center = (0, 0, 0)

# Create BEMSolver object
solver = cpt.BEMSolver()
dof_list = cptbody.active_dofs



xr = pyimport("xarray")
test_matrix = xr.Dataset(coords=Dict("omega" => omegas, "wave_direction" => [0.0], "radiating_dof" => DOFs))
results = cpt.BEMSolver().fill_dataset(test_matrix, cptbody, method="direct")



#  # Create DiffractionProblem object
#  diff_prob = cpt.DiffractionProblem(body=cptbody, wave_direction=beta, omega=omega, water_depth=h) 

#  # Create list of RadiationProblem objects
# rad_prob = [] 

# for dof in cptbody.active_dofs

#     prob = cpt.RadiationProblem(
#         body=cptbody, 
#         omega=omega, 
#         radiating_dof=dof, 
#         water_depth=h
#     )

#     push!(rad_prob, prob) # this is a Julia way of appending p to the list rad_prob
# end

[09:58:11] WARNING  The rotation dof Pitch has been initialized around the     
                    origin of the domain (0, 0, 0).                            
Solving BEM problems ---------------------------------------- 100% 0:00:00


PyObject <xarray.Dataset> Size: 700B
Dimensions:              (omega: 3, radiating_dof: 2, influenced_dof: 2,
                          wave_direction: 1)
Coordinates: (12/13)
  * omega                (omega) float64 24B 1.01 1.05 1.7
  * radiating_dof        (radiating_dof) category 18B Heave Pitch
  * influenced_dof       (influenced_dof) category 18B Heave Pitch
  * wave_direction       (wave_direction) float64 8B 0.0
    g                    float64 8B 9.81
    rho                  float64 8B 1e+03
    ...                   ...
    water_depth          float64 8B inf
    forward_speed        float64 8B 0.0
    freq                 (omega) float64 24B 0.1607 0.1671 0.2706
    period               (omega) float64 24B 6.221 5.984 3.696
    wavenumber           (omega) float64 24B 0.104 0.1124 0.2946
    wavelength           (omega) float64 24B 60.42 55.91 21.33
Data variables:
    added_mass           (omega, radiating_dof, influenced_dof) float64 96B 6...
    radiation_damping    (omega, radiating_dof, influenced_dof) float64 96B 1...
    diffraction_force    (omega, wave_direction, influenced_dof) complex128 96B ...
    Froude_Krylov_force  (omega, wave_direction, influenced_dof) complex128 96B ...
    excitation_force     (omega, wave_direction, influenced_dof) complex128 96B ...
Attributes: (12/18)
    start_of_computation:                     2026-02-19T09:58:11.245849
    green_function:                           Delhommeau
    tabulation_nr:                            676
    tabulation_rmax:                          100.0
    tabulation_nz:                            372
    tabulation_zmin:                          -251.0
    ...                                       ...
    engine:                                   BasicMatrixEngine
    matrix_cache_size:                        1
    linear_solver:                            lu_decomposition
    method:                                   direct
    creation_of_dataset:                      2026-02-19T09:58:11.261415
    capytaine_version:                        2.3.1

In [62]:
results.diffraction_force

PyObject <xarray.DataArray 'diffraction_force' (omega: 3, wave_direction: 1,
                                       influenced_dof: 2)> Size: 96B
array([[[ -6539.83589151-1791.58400421j,    -19.38550911+3244.42681619j]],

       [[ -6946.91530503-2018.40350079j,    -26.18430511+3512.51900866j]],

       [[-11852.5317173 -6795.65856077j,   -955.75385521+9330.33018731j]]])
Coordinates:
  * omega           (omega) float64 24B 1.01 1.05 1.7
  * wave_direction  (wave_direction) float64 8B 0.0
  * influenced_dof  (influenced_dof) category 18B Heave Pitch
    g               float64 8B 9.81
    rho             float64 8B 1e+03
    body_name       <U6 24B 'Body 1'
    water_depth     float64 8B inf
    forward_speed   float64 8B 0.0
    freq            (omega) float64 24B 0.1607 0.1671 0.2706
    period          (omega) float64 24B 6.221 5.984 3.696
    wavenumber      (omega) float64 24B 0.104 0.1124 0.2946
    wavelength      (omega) float64 24B 60.42 55.91 21.33
Attributes:
    long_name:  Diffraction force

In [59]:
# Get hydro_coeffs
num_dof = length(results.influenced_dof)
num_omega = length(omegas)
A_cpt = reshape(results.added_mass.values,num_dof,num_dof, num_omega)
B_cpt = reshape(results.radiation_damping.values,num_dof,num_dof, num_omega)
F_ex_cpt = reshape(results.excitation_force.values,num_dof,1,num_omega)

2×1×3 Array{ComplexF64, 3}:
[:, :, 1] =
 56847.384692542706 - 1791.5840042137793im
  55798.44750788946 - 2018.4035007923305im

[:, :, 2] =
    38111.1500342542 - 6795.658560768418im
 -19.385509110876228 + 5009.0871027415815im

[:, :, 3] =
 -26.184305108800118 + 5414.264710327202im
  -955.7538552105539 + 13915.874754426743im

## Create Multi DOF body using MarineHydro

In [8]:
# Create Mesh
mesh = Mesh(cptmesh)


Mesh([-1.25 -1.4623918682727355 -0.33378140093447134; -1.25 -1.4623918682727357 0.0; … ; 1.25 1.4623918682727355 -0.3337814009344717; 1.25 1.4623918682727355 -2.7755575615628914e-17], [21 33 32 12; 33 42 41 32; … ; 82 79 78 81; 59 56 55 58], [-0.9375 0.3254128043381684 -1.4257266509268143; -0.3125 0.3254128043381684 -1.4257266509268143; … ; 1.25 1.2349086887636433 -0.14092992483899916; 1.25 -1.2349086887636436 -0.14092992483899902], [0.0 0.2225209339563142 -0.9749279121818238; 0.0 0.2225209339563142 -0.9749279121818238; … ; 1.0 0.0 0.0; 1.0 0.0 0.0], [0.41722675116808966, 0.41722675116808966, 0.41722675116808966, 0.41722675116808966, 0.4172267511680895, 0.4172267511680895, 0.4172267511680895, 0.4172267511680895, 0.41722675116808955, 0.41722675116808955  …  0.054235467389694765, 0.02711773369484739, 0.02711773369484736, 0.05423546738969476, 0.05423546738969476, 0.05423546738969479, 0.08135320108454216, 0.08135320108454208, 0.13558866847423695, 0.13558866847423684], [0.45723765550288903,

In [9]:
# Create FloatingBody with custom dof matrices
num_panels = mesh.nfaces

dof_mat_1 = zeros(num_panels,3)
dof_mat_1[:,3].=1

dof_mat_2 = zeros(num_panels,3)
dof_mat_2[:,2].=1

dofs = Dict("DOF1"=>dof_mat_1, "DOF2"=>dof_mat_2)


fb = FloatingBody(mesh,dofs)

FloatingBody(Mesh([-1.25 -1.4623918682727355 -0.33378140093447134; -1.25 -1.4623918682727357 0.0; … ; 1.25 1.4623918682727355 -0.3337814009344717; 1.25 1.4623918682727355 -2.7755575615628914e-17], [21 33 32 12; 33 42 41 32; … ; 82 79 78 81; 59 56 55 58], [-0.9375 0.3254128043381684 -1.4257266509268143; -0.3125 0.3254128043381684 -1.4257266509268143; … ; 1.25 1.2349086887636433 -0.14092992483899916; 1.25 -1.2349086887636436 -0.14092992483899902], [0.0 0.2225209339563142 -0.9749279121818238; 0.0 0.2225209339563142 -0.9749279121818238; … ; 1.0 0.0 0.0; 1.0 0.0 0.0], [0.41722675116808966, 0.41722675116808966, 0.41722675116808966, 0.41722675116808966, 0.4172267511680895, 0.4172267511680895, 0.4172267511680895, 0.4172267511680895, 0.41722675116808955, 0.41722675116808955  …  0.054235467389694765, 0.02711773369484739, 0.02711773369484736, 0.05423546738969476, 0.05423546738969476, 0.05423546738969479, 0.08135320108454216, 0.08135320108454208, 0.13558866847423695, 0.13558866847423684], [0.45723

In [10]:
dofs["DOF1"]

80×3 Matrix{Float64}:
 0.0  0.0  1.0
 0.0  0.0  1.0
 0.0  0.0  1.0
 0.0  0.0  1.0
 0.0  0.0  1.0
 0.0  0.0  1.0
 0.0  0.0  1.0
 0.0  0.0  1.0
 0.0  0.0  1.0
 0.0  0.0  1.0
 ⋮         
 0.0  0.0  1.0
 0.0  0.0  1.0
 0.0  0.0  1.0
 0.0  0.0  1.0
 0.0  0.0  1.0
 0.0  0.0  1.0
 0.0  0.0  1.0
 0.0  0.0  1.0
 0.0  0.0  1.0

In [11]:
# Radiation BC
radiation_bc(fb, omega)

Dict{Any, Any} with 2 entries:
  "DOF1" => ComplexF64[-0.0+1.00418im; -0.0+1.00418im; … ; 0.0-0.0im; 0.0-0.0im…
  "DOF2" => ComplexF64[0.0-0.229197im; 0.0-0.229197im; … ; 0.0-0.0im; 0.0-0.0im…

In [12]:
# Integrate pressure
pressure = ones(mesh.nfaces,3) # dummy pressure
integrate_pressure(fb, pressure)

Dict{Any, Any} with 2 entries:
  "DOF1" => 21.9359
  "DOF2" => 4.44089e-16

In [13]:
A,B = calculate_radiation_forces(fb, omega)
A


Dict{Tuple{Float64, String, String}, Float64} with 4 entries:
  (1.03, "DOF2", "DOF2") => 5443.62
  (1.03, "DOF1", "DOF2") => 2.06959e-12
  (1.03, "DOF2", "DOF1") => 1.54044e-12
  (1.03, "DOF1", "DOF1") => 6978.06

In [14]:
F_D = DiffractionForce(fb,omega)

Dict{Tuple{Float64, String}, ComplexF64} with 2 entries:
  (1.03, "DOF2") => 2.27374e-13+2.23821e-13im
  (1.03, "DOF1") => -6746.09-1904.15im

In [15]:
F_FK = FroudeKrylovForce(fb, omega)

Dict{Tuple{Float64, String}, ComplexF64} with 2 entries:
  (1.03, "DOF2") => 1.36424e-12-8.52651e-14im
  (1.03, "DOF1") => 63068.8+1.47096e-13im

In [28]:
rigid_dof_list = ["Heave"]
rotation_center = [0,0,0]
fb2 = FloatingBody(mesh, rigid_dof_list, rotation_center)

FloatingBody(Mesh([-1.25 -1.4623918682727355 -0.33378140093447134; -1.25 -1.4623918682727357 0.0; … ; 1.25 1.4623918682727355 -0.3337814009344717; 1.25 1.4623918682727355 -2.7755575615628914e-17], [21 33 32 12; 33 42 41 32; … ; 82 79 78 81; 59 56 55 58], [-0.9375 0.3254128043381684 -1.4257266509268143; -0.3125 0.3254128043381684 -1.4257266509268143; … ; 1.25 1.2349086887636433 -0.14092992483899916; 1.25 -1.2349086887636436 -0.14092992483899902], [0.0 0.2225209339563142 -0.9749279121818238; 0.0 0.2225209339563142 -0.9749279121818238; … ; 1.0 0.0 0.0; 1.0 0.0 0.0], [0.41722675116808966, 0.41722675116808966, 0.41722675116808966, 0.41722675116808966, 0.4172267511680895, 0.4172267511680895, 0.4172267511680895, 0.4172267511680895, 0.41722675116808955, 0.41722675116808955  …  0.054235467389694765, 0.02711773369484739, 0.02711773369484736, 0.05423546738969476, 0.05423546738969476, 0.05423546738969479, 0.08135320108454216, 0.08135320108454208, 0.13558866847423695, 0.13558866847423684], [0.45723

In [29]:
A,B = calculate_radiation_forces(fb2, omega)
A

Dict{Tuple{Float64, String, String}, Float64} with 1 entry:
  (1.03, "Heave", "Heave") => 6978.06

In [30]:
F_D = DiffractionForce(fb2,omega)

Dict{Tuple{Float64, String}, ComplexF64} with 1 entry:
  (1.03, "Heave") => -6746.09-1904.15im

In [31]:
F_FK = FroudeKrylovForce(fb2, omega)

Dict{Tuple{Float64, String}, ComplexF64} with 1 entry:
  (1.03, "Heave") => 63068.8+1.47096e-13im

In [32]:
calculate_radiation_forces(fb2, omega)[1][(omega, "Heave", "Heave")]

6978.062210908977

In [33]:
# Try gradient with new MDOF functions
using Zygote
B_w_grad, = Zygote.gradient(w -> calculate_radiation_forces(fb2, w)[2][(omega, "Heave", "Heave")],omega)

(3663.0568518277705,)

In [34]:
# Gradient with old SDOF functions
ζ = [0,0,1]
B_w_grad, = Zygote.gradient(w -> calculate_radiation_forces(mesh,ζ,w)[2],omega)

(3663.2346440990673,)

In [23]:
100*(1945-1942)/1942

0.15447991761071062

In [24]:
A_cpt

2×2 Matrix{Float64}:
 6818.24           -8.57286e-13
   -4.28038e-13  3133.43

In [25]:
A

Dict{Tuple{Float64, String, String}, Float64} with 4 entries:
  (1.03, "Pitch", "Heave") => 1.03812e-13
  (1.03, "Heave", "Heave") => 6978.06
  (1.03, "Heave", "Pitch") => -5.35188e-13
  (1.03, "Pitch", "Pitch") => 3204.74

In [26]:
B_cpt

2×2 Matrix{Float64}:
 1825.11         0.0
    7.92201e-15  8.3114

In [27]:
B

Dict{Tuple{Float64, String, String}, Float64} with 4 entries:
  (1.03, "Pitch", "Heave") => -3.24228e-13
  (1.03, "Heave", "Heave") => 1866.89
  (1.03, "Heave", "Pitch") => -3.98247e-14
  (1.03, "Pitch", "Pitch") => 8.49764